In [13]:
import json
import math
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz

In [14]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [15]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [16]:
def is_similar(text1, text2, threshold=80):
    return fuzz.partial_ratio(text1.strip(), text2.strip()) >= threshold

In [17]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [18]:
reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3')

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5236.67it/s]


In [19]:
def get_reranked_results(query, initial_results, k=5):
    points = initial_results.points
    pairs = [[query, res.payload['text']] for res in points]
    scores = reranker_model.predict(pairs)
    scored_results = sorted(zip(points, scores), key=lambda x: x[1], reverse=True)

    return [res[0] for res in scored_results[:k]]

In [20]:
def evaluate_retrieval_metrics(search_results, golden_context, k=5):
    hit = 0
    reciprocal_rank = 0
    relevant_found_count = 0
    dcg = 0
    
    for rank, result in enumerate(search_results[:k], start=1):
        if any(is_similar(golden_text, result) for golden_text in golden_context):
            if hit == 0:
                hit = 1
                reciprocal_rank = 1 / rank
            relevant_found_count += 1
            dcg += 1 / math.log2(rank + 1)
    recall = relevant_found_count / len(golden_context) if len(golden_context) > 0 else 0
    idcg = sum(1 / math.log2(i + 1) for i in range(1, min(len(golden_context), k) + 1))
    ndcg = dcg / idcg if idcg > 0 else 0
    return hit, reciprocal_rank, recall, ndcg

In [21]:
def get_metrics(model, collection, e5=False):
    total_hit = 0
    total_mrr = 0
    total_recall = 0
    total_ndcg = 0

    total_hit_reranked = 0
    total_mrr_reranked = 0
    total_recall_reranked = 0
    total_ndcg_reranked = 0

    for item in golden_data:
        if e5:
            query = model.encode("query: " + item['input']).tolist()
        else:
            query=model.encode(item['input']).tolist()
        results = client.query_points(
            collection_name=collection,
            query=query,
            limit=20
        )
        found_texts = [res.payload['text'] for res in results.points]
        
        h, m, r, n = evaluate_retrieval_metrics(found_texts, item['retrieval_context'])
        total_hit += h
        total_mrr += m
        total_recall += r
        total_ndcg += n

        reranked_objects = get_reranked_results(item['input'], results)
        reranked_texts = [res.payload['text'] for res in reranked_objects]

        h_rerank, m_rerank, r_rerank, n_rerank = evaluate_retrieval_metrics(reranked_texts, item['retrieval_context'])
        total_hit_reranked += h_rerank
        total_mrr_reranked += m_rerank
        total_recall_reranked += r_rerank
        total_ndcg_reranked += n_rerank

    return total_hit, total_mrr, total_recall, total_ndcg, \
            total_hit_reranked, total_mrr_reranked, total_recall_reranked, total_ndcg_reranked

In [22]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
COLLECTION_BASELINE = "ucu_documents_baseline"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6148.37it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
total_hit_baseline, total_mrr_baseline, total_recall_baseline, total_ndcg_baseline, \
total_hit_reranked_baseline, total_mrr_reranked_baseline, total_recall_reranked_baseline, total_ndcg_reranked_baseline = get_metrics(MODEL_BASELINE, COLLECTION_BASELINE)

In [26]:
n = len(golden_data)

In [ ]:
print("Results for baseline model:")
print(f"Hit Rate @ 5: {round(total_hit_baseline/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline/n, 2)}")

Results for baseline model:
Hit Rate @ 5: 0.5
MRR @ 5: 0.31
Recall @ 5: 0.42
NDCG @ 5: 0.32


In [30]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [31]:
total_hit_baseline_2, total_mrr_baseline_2, total_recall_baseline_2, total_ndcg_baseline_2, \
total_hit_reranked_baseline_2, total_mrr_reranked_baseline_2, total_recall_reranked_baseline_2, total_ndcg_reranked_baseline_2 = get_metrics(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR)

In [32]:
print("Results for baseline model (different chunking):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_2/n, 2)}")

Results for baseline model (different chunking):
Hit Rate @ 5: 0.45
MRR @ 5: 0.3
Recall @ 5: 0.47
NDCG @ 5: 0.36


In [33]:
print("Results for baseline model (different chunking + reranker):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_2/n, 2)}")

Results for baseline model (different chunking + reranker):
Hit Rate @ 5: 0.7
MRR @ 5: 0.64
Recall @ 5: 0.75
NDCG @ 5: 0.68
